## Cell 1: imports and setup ##

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(exist_ok=True)

print("Folders ready. Starting cleaning pipeline.")

Folders ready. Starting cleaning pipeline.


## Cell 2: load Olist tables ##

In [2]:
olist_dir = RAW / "olist"

orders = pd.read_csv(olist_dir / "olist_orders_dataset.csv")
order_items = pd.read_csv(olist_dir / "olist_order_items_dataset.csv")
reviews = pd.read_csv(olist_dir / "olist_order_reviews_dataset.csv")
customers = pd.read_csv(olist_dir / "olist_customers_dataset.csv")
payments = pd.read_csv(olist_dir / "olist_order_payments_dataset.csv")
products = pd.read_csv(olist_dir / "olist_products_dataset.csv")

print("Loaded:")
for name, df in [("orders", orders), ("order_items", order_items), ("reviews", reviews),
                 ("customers", customers), ("payments", payments), ("products", products)]:
    print(f"  {name:14s} {df.shape}")

Loaded:
  orders         (99441, 8)
  order_items    (112650, 7)
  reviews        (99224, 7)
  customers      (99441, 5)
  payments       (103886, 5)
  products       (32951, 9)


## Cell 3: standardize dates in orders ##

In [3]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

print("Date columns converted to datetime")
print("\nMissing values per date column:")
print(orders[date_cols].isna().sum())

Date columns converted to datetime

Missing values per date column:
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


## Cell 4: order status distribution ##

In [4]:
print("Order status counts:")
print(orders["order_status"].value_counts())

Order status counts:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


## Cell 5: derived columns on orders ##

In [5]:
# delivery time in days (purchase to customer received)
orders["delivery_time_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

# days late vs estimated (negative = early, positive = late)
orders["days_vs_estimate"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days

# binary flag for late delivery
orders["is_late"] = orders["days_vs_estimate"] > 0

# only on delivered orders, what's the average?
delivered = orders[orders["order_status"] == "delivered"]
print(f"Delivered orders: {len(delivered):,}")
print(f"Avg delivery time: {delivered['delivery_time_days'].mean():.1f} days")
print(f"Late delivery rate: {delivered['is_late'].mean() * 100:.1f}%")
print(f"Avg days vs estimate (when late): {delivered.loc[delivered['is_late'], 'days_vs_estimate'].mean():.1f} days")

Delivered orders: 96,478
Avg delivery time: 12.1 days
Late delivery rate: 6.8%
Avg days vs estimate (when late): 10.6 days


## Cell 6: clean reviews ##

In [6]:
# convert review timestamps
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"], errors="coerce")
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"], errors="coerce")

# how long until the seller responded (in days)
reviews["response_time_days"] = (
    reviews["review_answer_timestamp"] - reviews["review_creation_date"]
).dt.days

# duplicate check
print(f"Total reviews: {len(reviews):,}")
print(f"Unique review IDs: {reviews['review_id'].nunique():,}")
print(f"Duplicate review IDs: {reviews['review_id'].duplicated().sum():,}")

# drop any duplicates if they exist
reviews = reviews.drop_duplicates(subset="review_id", keep="first")
print(f"After dedup: {len(reviews):,}")

Total reviews: 99,224
Unique review IDs: 98,410
Duplicate review IDs: 814
After dedup: 98,410


## Cell 7: aggregate payments per order ##

In [7]:
payments_agg = payments.groupby("order_id").agg(
    payment_count=("payment_sequential", "max"),
    total_payment=("payment_value", "sum"),
    payment_types=("payment_type", lambda x: ",".join(sorted(set(x)))),
    main_payment_type=("payment_type", lambda x: x.value_counts().idxmax()),
).reset_index()

print(payments_agg.head())
print(f"\nPayment type breakdown:")
print(payments_agg["main_payment_type"].value_counts())

                           order_id  payment_count  total_payment payment_types main_payment_type
0  00010242fe8c5a6d1ba2dd792cb16214              1          72.19   credit_card       credit_card
1  00018f77f2f0320c557190d7a144bdd3              1         259.83   credit_card       credit_card
2  000229ec398224ef6ca0657da4fc703e              1         216.87   credit_card       credit_card
3  00024acbcdf0a6daa1e931b038114c75              1          25.78   credit_card       credit_card
4  00042b26cf59d7ce69dfabb4e55b4fd9              1         218.04   credit_card       credit_card

Payment type breakdown:
main_payment_type
credit_card    75270
boleto         19784
voucher         2856
debit_card      1527
not_defined        3
Name: count, dtype: int64


## Cell 8: aggregate order items per order ##

In [8]:
items_agg = order_items.groupby("order_id").agg(
    item_count=("order_item_id", "max"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    distinct_products=("product_id", "nunique"),
).reset_index()

print(items_agg.head())

                           order_id  item_count  total_price  total_freight  distinct_products
0  00010242fe8c5a6d1ba2dd792cb16214           1        58.90          13.29                  1
1  00018f77f2f0320c557190d7a144bdd3           1       239.90          19.93                  1
2  000229ec398224ef6ca0657da4fc703e           1       199.00          17.87                  1
3  00024acbcdf0a6daa1e931b038114c75           1        12.99          12.79                  1
4  00042b26cf59d7ce69dfabb4e55b4fd9           1       199.90          18.14                  1


## Cell 9: build the master dataframe ##

In [9]:
master = (
    orders
    .merge(reviews, on="order_id", how="left")
    .merge(customers, on="customer_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(items_agg, on="order_id", how="left")
)

print(f"Master shape: {master.shape}")
print(f"\nColumns: {master.columns.tolist()}")

Master shape: (99684, 30)

Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_time_days', 'days_vs_estimate', 'is_late', 'review_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp', 'response_time_days', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'payment_count', 'total_payment', 'payment_types', 'main_payment_type', 'item_count', 'total_price', 'total_freight', 'distinct_products']


## Cell 10: sanity check on master ##

In [10]:
print("Missing values (top 15 cols):")
print(master.isna().sum().sort_values(ascending=False).head(15))

print(f"\nOrders with reviews: {master['review_id'].notna().sum():,}")
print(f"Orders with payments: {master['total_payment'].notna().sum():,}")
print(f"Orders with items: {master['item_count'].notna().sum():,}")

Missing values (top 15 cols):
review_comment_title             88165
review_comment_message           59016
order_delivered_customer_date     2971
delivery_time_days                2971
days_vs_estimate                  2971
order_delivered_carrier_date      1784
review_creation_date              1274
response_time_days                1274
review_answer_timestamp           1274
review_id                         1274
review_score                      1274
total_freight                      775
total_price                        775
item_count                         775
distinct_products                  775
dtype: int64

Orders with reviews: 98,410
Orders with payments: 99,683
Orders with items: 98,909


In [11]:
# sanity check: did the merge multiply rows?
print(f"Original orders shape: {orders.shape}")
print(f"Master shape: {master.shape}")
print(f"Unique order_ids in master: {master['order_id'].nunique():,}")

Original orders shape: (99441, 11)
Master shape: (99684, 30)
Unique order_ids in master: 99,441


In [12]:
# check uniqueness of order_id in each side table
for name, df in [("reviews", reviews), ("payments_agg", payments_agg), ("items_agg", items_agg)]:
    total = len(df)
    unique = df["order_id"].nunique()
    dupes = total - unique
    print(f"{name:14s} rows={total:,}  unique order_ids={unique:,}  duplicates={dupes}")

reviews        rows=98,410  unique order_ids=98,167  duplicates=243
payments_agg   rows=99,440  unique order_ids=99,440  duplicates=0
items_agg      rows=98,666  unique order_ids=98,666  duplicates=0


In [13]:
# find the orders with duplicate reviews
dup_order_ids = reviews[reviews.duplicated(subset="order_id", keep=False)]["order_id"].unique()
print(f"Orders with multiple reviews: {len(dup_order_ids)}")

# show a sample
sample_order = dup_order_ids[0]
print(f"\nExample: order {sample_order}")
print(reviews[reviews["order_id"] == sample_order][["review_id", "review_score", "review_creation_date", "review_comment_message"]])

Orders with multiple reviews: 243

Example: order cf73e2cb1f4a9480ed70c154da3d954a
                             review_id  review_score review_creation_date review_comment_message
30    540e7bbb2d06cfb7f85f3a88ba7ac97f             5           2018-01-18                    NaN
3109  aa193e76d35950c4ae988237bb36ed2b             5           2018-01-18                    NaN


In [14]:
reviews_clean = (
    reviews
    .sort_values("review_creation_date", ascending=False)
    .drop_duplicates(subset="order_id", keep="first")
    .reset_index(drop=True)
)

print(f"Before dedup: {len(reviews):,}")
print(f"After dedup:  {len(reviews_clean):,}")
print(f"Unique order_ids: {reviews_clean['order_id'].nunique():,}")

Before dedup: 98,410
After dedup:  98,167
Unique order_ids: 98,167


In [15]:
master = (
    orders
    .merge(reviews_clean, on="order_id", how="left")
    .merge(customers, on="customer_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(items_agg, on="order_id", how="left")
)

print(f"Master shape: {master.shape}")
print(f"Unique order_ids: {master['order_id'].nunique():,}")
print(f"Match orders count? {master.shape[0] == orders.shape[0]}")

Master shape: (99441, 30)
Unique order_ids: 99,441
Match orders count? True


In [16]:
master.to_parquet(PROCESSED / "olist_master.parquet", index=False)
print(f"Saved master: {PROCESSED / 'olist_master.parquet'}")
print(f"File size: {(PROCESSED / 'olist_master.parquet').stat().st_size / 1024 / 1024:.1f} MB")

Saved master: ..\data\processed\olist_master.parquet
File size: 20.7 MB


# Load Women's Clothing data #

In [17]:
womens_path = list((RAW / "womens_clothing").glob("*.csv"))[0]
womens = pd.read_csv(womens_path)

print(f"Shape: {womens.shape}")
print(f"Columns: {womens.columns.tolist()}")
womens.head(2)

Shape: (23486, 11)
Columns: ['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name']


,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses


# drop rows with no review text #

In [18]:
print(f"Before drop: {len(womens):,}")
print(f"Rows with null Review Text: {womens['Review Text'].isna().sum():,}")

womens_text = womens[womens["Review Text"].notna()].reset_index(drop=True)
print(f"After drop:  {len(womens_text):,}")

Before drop: 23,486
Rows with null Review Text: 845
After drop:  22,641


# some raw reviews so we know what we're cleaning #

In [19]:
print("Sample raw reviews:\n")
for i, review in enumerate(womens_text["Review Text"].sample(5, random_state=42)):
    print(f"--- Review {i+1} ---")
    print(review)
    print()

Sample raw reviews:

--- Review 1 ---
This sweater is so beautiful on. it is thick material, but does not make you look boxy. it fits so nicely and is flattering. the design is just gorgeous. if you are considering this sweater-- get it, you won't regret it!

i'm normally a small (4-6) and i ordered a small. it fits true to size. the fit is *perfect* so if you want it slightly more relaxed, order one size up.

--- Review 2 ---
This piece is almost what i want... i tried on the white version in an xs and it felt a little too large for me, even though it's meant to be a looser fit. the buttercup yellow behind the lace is very pretty - i especially love that part of this tank. it provides a nice contrast and allows the detail to stand out. the navy is also nice (it was on display next to the white and looks better in person than online). what i didn't like was the texture of the spandex on the back portion. it looks like

--- Review 3 ---
Really like this blouse but am returning for a lar

In [20]:
import re

# Define patterns we want to hunt for
patterns = {
    "HTML tags":               r"<[^>]+>",
    "URLs":                    r"http\S+|www\.\S+",
    "Email addresses":         r"\S+@\S+",
    "Emojis (basic)":          r"[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF\u2600-\u26FF\u2700-\u27BF]",
    "Repeated chars (3+)":     r"(.)\1{2,}",          # e.g. "soooo", "!!!!"
    "All caps words (4+)":     r"\b[A-Z]{4,}\b",      # SHOUTING
    "Multiple exclamations":   r"!{2,}",
    "Numbers":                 r"\d+",
    "Special chars":           r"[^a-zA-Z0-9\s.,!?'\"-]",
    "Doubled apostrophes":     r"'{2,}",
    "Backslash sequences":     r"\\[a-z]",            # \n, \t etc. that didn't get parsed
    "Non-ASCII chars":         r"[^\x00-\x7F]",
}

print(f"Checking {len(womens_text):,} reviews for noise patterns:\n")
print(f"{'Pattern':<25} {'Count':>8} {'%':>6}")
print("-" * 42)
for name, pattern in patterns.items():
    matches = womens_text["Review Text"].str.contains(pattern, regex=True, na=False).sum()
    pct = matches / len(womens_text) * 100
    print(f"{name:<25} {matches:>8,} {pct:>5.1f}%")

Checking 22,641 reviews for noise patterns:

Pattern                      Count      %
------------------------------------------
HTML tags                        0   0.0%
URLs                             0   0.0%
Email addresses                  0   0.0%
Emojis (basic)                   0   0.0%
Repeated chars (3+)          2,629  11.6%
All caps words (4+)              0   0.0%
Multiple exclamations          982   4.3%
Numbers                      8,790  38.8%
Special chars                7,639  33.7%
Doubled apostrophes            138   0.6%
Backslash sequences              0   0.0%
Non-ASCII chars                 30   0.1%


In [21]:
def show_examples(pattern, name, n=3):
    """Pull a few reviews matching a noise pattern so we can see what we're dealing with."""
    matches = womens_text[womens_text["Review Text"].str.contains(pattern, regex=True, na=False)]
    if len(matches) == 0:
        print(f"\n=== {name} ===\nNo matches.")
        return
    print(f"\n=== {name} ({len(matches):,} reviews) ===")
    for i, txt in enumerate(matches["Review Text"].sample(min(n, len(matches)), random_state=42).values):
        print(f"[{i+1}] {txt[:250]}")
        print()

# show examples of each noisy pattern that had hits
for name, pattern in patterns.items():
    show_examples(pattern, name, n=2)


=== HTML tags ===
No matches.

=== URLs ===
No matches.

=== Email addresses ===
No matches.

=== Emojis (basic) ===
No matches.

=== Repeated chars (3+) (2,629 reviews) ===
[1] Classic and stylish. trendy with the blocking...rich colors, soft and comfy. i typically wear s and s was perfect. i love this sweater.

[2] I absolutely love these shorts!!! everything about them is perfect!! i have them in almost every color. they wash beautifully. i don't dry my shorts in the dryer. they fit the same as last years'. i'm thin & pretty straight with little hips. i wear t


=== All caps words (4+) ===
No matches.

=== Multiple exclamations (982 reviews) ===
[1] Wonderful colors and fit tts. love the style. can were on or off the shoulders . love it!!!

[2] The details on this top are absolutely gorgeous, from the lace, to the sequins, to the rhinestones! that said, the details are not overpowering, yet more subtle. can't wait to wear this over the holiday season. sure to get tons of compliment

In [22]:
# review length distribution — extreme short/long reviews are usually noise
womens_text["char_length"] = womens_text["Review Text"].str.len()
womens_text["word_count"] = womens_text["Review Text"].str.split().str.len()

print("Character length distribution:")
print(womens_text["char_length"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

print("\nWord count distribution:")
print(womens_text["word_count"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

print(f"\nReviews with fewer than 5 words: {(womens_text['word_count'] < 5).sum():,}")
print(f"Reviews with fewer than 10 chars: {(womens_text['char_length'] < 10).sum():,}")
print(f"Reviews longer than 2000 chars:   {(womens_text['char_length'] > 2000).sum():,}")

Character length distribution:
count    22641.000000
mean       308.687911
std        143.940048
min          9.000000
1%          52.000000
5%          84.000000
50%        301.000000
95%        500.000000
99%        502.000000
max        508.000000
Name: char_length, dtype: float64

Word count distribution:
count    22641.000000
mean        60.196679
std         28.534612
min          2.000000
1%           9.000000
5%          16.000000
50%         59.000000
95%        101.000000
99%        106.000000
max        115.000000
Name: word_count, dtype: float64

Reviews with fewer than 5 words: 34
Reviews with fewer than 10 chars: 1
Reviews longer than 2000 chars:   0


In [23]:
print("=== 5 SHORTEST reviews ===")
for txt in womens_text.nsmallest(5, "char_length")["Review Text"]:
    print(f"  {repr(txt)}")

print("\n=== 3 LONGEST reviews (first 300 chars) ===")
for txt in womens_text.nlargest(3, "char_length")["Review Text"]:
    print(f"  {txt[:300]}...")

=== 5 SHORTEST reviews ===
  'Great fit'
  'Comfy cozy!'
  'Great style!'
  'I love birds'
  'Fits perfect!'

=== 3 LONGEST reviews (first 300 chars) ===
  I adore this blouse. the colors are vibrant (see my photo below). this is one of my favorite purchases from retailer. the top is light weight. true to size. i ordered a petite small and am 5 feet tall 120 lbs. and curvy. i left it untucked and loose like in the photo and it was very flattering. i di...
  I have been continually disappointed in retailer's clothing for the past year. i feel like their prices have soared and the quality has remained so-so, but dresses like these somehow make the cut. it does not make sense. since when did retailer exclusively market to older women? no, retailer's marke...
  I love this cotton weave shift dress. it is supposed to be loose and not tight. i purchased a medium and fits the way i see it should fit. ref: 34dd, 148 lb; 5'4 height. hits above the knee but office appropriate as not too short. i

## light cleaning function ##

In [24]:
import re
import unicodedata

def clean_text(text):
    """Light cleaning: removes encoding artifacts, normalizes inch notation,
    collapses whitespace. PRESERVES numbers, punctuation, case (mostly), 
    and stylistic emphasis like '...' and '!!!'.
    
    Designed for downstream models (DistilBERT, BERTopic) that handle their
    own tokenization and benefit from rich text.
    """
    if not isinstance(text, str):
        return ""
    # normalize unicode (handles 'ombré' -> 'ombre' or similar mangling)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    # fix doubled apostrophes (inch notation: 5'2'' -> 5'2')
    text = re.sub(r"'{2,}", "'", text)
    # collapse multiple spaces, tabs, newlines into single space
    text = re.sub(r"\s+", " ", text).strip()
    return text

# test on a few problematic samples
test_cases = [
    "i'm 5'2'' and a curvy 135 pounds",
    "an ombrã© look to it fading down to darker",
    "love it!!! perfect!!  multiple   spaces  here",
    "size 29 did not fit",
]
for t in test_cases:
    print(f"BEFORE: {t}")
    print(f"AFTER:  {clean_text(t)}\n")

BEFORE: i'm 5'2'' and a curvy 135 pounds
AFTER:  i'm 5'2' and a curvy 135 pounds

BEFORE: an ombrã© look to it fading down to darker
AFTER:  an ombra look to it fading down to darker

BEFORE: love it!!! perfect!!  multiple   spaces  here
AFTER:  love it!!! perfect!! multiple spaces here

BEFORE: size 29 did not fit
AFTER:  size 29 did not fit



## apply to all reviews + flag short ones ##

In [25]:
womens_text["review_clean"] = womens_text["Review Text"].apply(clean_text)
womens_text["is_short"] = womens_text["word_count"] < 5

print(f"Total reviews:     {len(womens_text):,}")
print(f"Short reviews:     {womens_text['is_short'].sum():,} ({womens_text['is_short'].mean()*100:.2f}%)")
print(f"Usable for themes: {(~womens_text['is_short']).sum():,}")

# spot check a few cleanings
print("\n=== Cleaning spot check ===")
for orig, clean in womens_text[["Review Text", "review_clean"]].sample(3, random_state=42).values:
    print(f"ORIG : {orig[:150]}")
    print(f"CLEAN: {clean[:150]}\n")

Total reviews:     22,641
Short reviews:     34 (0.15%)
Usable for themes: 22,607

=== Cleaning spot check ===
ORIG : This sweater is so beautiful on. it is thick material, but does not make you look boxy. it fits so nicely and is flattering. the design is just gorgeo
CLEAN: This sweater is so beautiful on. it is thick material, but does not make you look boxy. it fits so nicely and is flattering. the design is just gorgeo

ORIG : This piece is almost what i want... i tried on the white version in an xs and it felt a little too large for me, even though it's meant to be a looser
CLEAN: This piece is almost what i want... i tried on the white version in an xs and it felt a little too large for me, even though it's meant to be a looser

ORIG : Really like this blouse but am returning for a larger size. much too tight in the upper arms and somewhat tight in the chest - overall i like it.
CLEAN: Really like this blouse but am returning for a larger size. much too tight in the upper arms an

## aggressive preprocessing function (for ML features) ##

In [26]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """Aggressive cleaning for TF-IDF, topic modeling, and bag-of-words features.
    Lowercases, strips punctuation/numbers, removes stopwords, lemmatizes.
    Returns space-separated tokens.
    """
    if not isinstance(text, str) or not text.strip():
        return ""
    text = text.lower()
    # keep only letters and spaces (numbers and punctuation gone)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    # remove stopwords, very short tokens, lemmatize the rest
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return " ".join(tokens)

# test
sample = "I'm 5'2'' and the size 6 fit was perfect! Loved the soft fabric."
print(f"BEFORE: {sample}")
print(f"AFTER:  {preprocess_text(sample)}")

BEFORE: I'm 5'2'' and the size 6 fit was perfect! Loved the soft fabric.
AFTER:  size fit perfect loved soft fabric


##  apply aggressive preprocessing to all reviews ##

In [27]:
from tqdm import tqdm
tqdm.pandas(desc="Preprocessing")

womens_text["review_processed"] = womens_text["review_clean"].progress_apply(preprocess_text)

# spot check
print("\n=== Clean vs Processed ===")
for clean, proc in womens_text[["review_clean", "review_processed"]].sample(3, random_state=11).values:
    print(f"CLEAN:     {clean[:130]}")
    print(f"PROCESSED: {proc[:130]}\n")

Preprocessing: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 22641/22641 [00:03<00:00, 7303.22it/s]


=== Clean vs Processed ===
CLEAN:     This is a very stylish cardigan but it runs small. i usually take a small but i took a medium in this. for reference i am 130 lbs,
PROCESSED: stylish cardigan run small usually take small took medium reference lb chest fell love put watch snag extremely delicate got today

CLEAN:     I love western motif so i had to try this t shirt. quality is great..material is high quality. also the length is long..perfect wi
PROCESSED: love western motif try shirt quality great material high quality also length long perfect jean jacket sweater

CLEAN:     Cute and perhaps a little too innocent for my age because the petite was too high and wide on me. fits great on my shoulders/chest
PROCESSED: cute perhaps little innocent age petite high wide fit great shoulder chest flare controlled layer belt already ivory lace dress lo



In [28]:
print(f"Total reviews:                    {len(womens_text):,}")
print(f"Empty review_clean:               {(womens_text['review_clean'] == '').sum():,}")
print(f"Empty review_processed:           {(womens_text['review_processed'] == '').sum():,}")
print(f"\nAverage tokens after preprocessing: {womens_text['review_processed'].str.split().str.len().mean():.1f}")
print(f"Min/Max processed tokens:           {womens_text['review_processed'].str.split().str.len().min()}/{womens_text['review_processed'].str.split().str.len().max()}")

Total reviews:                    22,641
Empty review_clean:               0
Empty review_processed:           0

Average tokens after preprocessing: 27.8
Min/Max processed tokens:           2/57


In [29]:
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

# make langdetect deterministic (it has some randomness by default)
DetectorFactory.seed = 42

def safe_detect(text):
    """Detect language, return 'unknown' if detection fails (very short text, etc.)"""
    if not isinstance(text, str) or len(text.strip()) < 10:
        return "unknown"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"

# test on a few samples
print("=== langdetect smoke test ===")
samples = [
    "This product is amazing, very happy with my purchase",
    "Produto chegou rápido e dentro do prazo, recomendo",
    "Muy bueno el producto",
    "ok",
]
for s in samples:
    print(f"{safe_detect(s):8s} | {s}")

=== langdetect smoke test ===
en       | This product is amazing, very happy with my purchase
pt       | Produto chegou rápido e dentro do prazo, recomendo
es       | Muy bueno el producto
unknown  | ok


In [30]:
# get only Olist reviews that have text to detect on
olist_with_text = reviews_clean[reviews_clean["review_comment_message"].notna()].copy()
print(f"Olist reviews with text: {len(olist_with_text):,}")

# run language detection (this takes ~1-2 min)
tqdm.pandas(desc="Detecting language")
olist_with_text["language"] = olist_with_text["review_comment_message"].progress_apply(safe_detect)

# show breakdown
print("\nLanguage distribution:")
print(olist_with_text["language"].value_counts().head(10))

Olist reviews with text: 40,577


Detecting language: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40577/40577 [03:55<00:00, 172.50it/s]


Language distribution:
language
pt         34258
unknown     3179
it          1207
es           725
ro           193
sk           178
en           159
de           130
ca           122
sl            80
Name: count, dtype: int64


## merge language info back into master ##

In [31]:
# bring language back to reviews_clean so it carries through
reviews_clean = reviews_clean.merge(
    olist_with_text[["review_id", "language"]],
    on="review_id",
    how="left"
)

# rebuild master with the language info attached
master = (
    orders
    .merge(reviews_clean, on="order_id", how="left")
    .merge(customers, on="customer_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(items_agg, on="order_id", how="left")
)

print(f"Master shape: {master.shape}")
print(f"Reviews with language detected: {master['language'].notna().sum():,}")
print(f"\nLanguage breakdown in master:")
print(master["language"].value_counts().head(8))

Master shape: (99441, 31)
Reviews with language detected: 40,577

Language breakdown in master:
language
pt         34258
unknown     3179
it          1207
es           725
ro           193
sk           178
en           159
de           130
Name: count, dtype: int64


## save processed datasets ##

In [32]:
# save the cleaned, joined Olist master
master.to_parquet(PROCESSED / "olist_master.parquet", index=False)
size_mb = (PROCESSED / "olist_master.parquet").stat().st_size / 1024 / 1024
print(f"Saved olist_master.parquet ({size_mb:.1f} MB)")

# save the cleaned Women's Clothing reviews (with both cleaning columns)
womens_text.to_parquet(PROCESSED / "womens_clean.parquet", index=False)
size_mb = (PROCESSED / "womens_clean.parquet").stat().st_size / 1024 / 1024
print(f"Saved womens_clean.parquet ({size_mb:.1f} MB)")

# save the language-tagged Olist reviews subset
olist_with_text.to_parquet(PROCESSED / "olist_reviews_with_lang.parquet", index=False)
size_mb = (PROCESSED / "olist_reviews_with_lang.parquet").stat().st_size / 1024 / 1024
print(f"Saved olist_reviews_with_lang.parquet ({size_mb:.1f} MB)")

Saved olist_master.parquet (20.7 MB)
Saved womens_clean.parquet (10.5 MB)
Saved olist_reviews_with_lang.parquet (4.8 MB)


## final sanity check ## 

In [33]:
# confirm everything's saved and readable
import os

print("Files in data/processed/:")
for f in PROCESSED.iterdir():
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:40s} {size_mb:>6.2f} MB")

# read one back to confirm it works
test = pd.read_parquet(PROCESSED / "olist_master.parquet")
print(f"\nRead back olist_master.parquet: {test.shape}")
print(f"Columns: {len(test.columns)}")

Files in data/processed/:
  .gitkeep                                   0.00 MB
  olist_master.parquet                      20.74 MB
  olist_reviews_with_lang.parquet            4.81 MB
  womens_clean.parquet                      10.46 MB

Read back olist_master.parquet: (99441, 31)
Columns: 31
